# Measuring fan blade width with a laser and a photodiode

This notebook puts the ideas from `nidaqmx_tutorial.ipynb` into a small experiment.
A laser beam passes through a spinning fan onto a photodiode. Each time a blade passes,
it blocks the beam and the photodiode signal drops. From the recorded signal we measure:

* the **pulse width** τ, i.e. how long each blade blocks the beam
* the **blade period** T, i.e. the time between one blade and the next

From these, together with the number of blades and where the beam crosses the fan, we get the blade width.

```
          laser ───────────────► ┃ ◄─ fan blade ─► photodiode ──► AI0
                                  (beam at radius r from the hub)

 photodiode   ─────┐     ┌─────────┐     ┌─────────┐     ┌────
 signal            │     │         │     │         │     │
                   └─────┘         └─────┘         └─────┘
                   ◄─ τ ─►
                   ◄────── T ──────►
```

### The physics

The fan has `N` blades, so it turns once every `N·T` seconds. Its rotation frequency is

$$f = \frac{1}{N\,T}$$

At radius $r$ the blade moves at $v = 2\pi r f$, so the distance swept while the beam is blocked is

$$w_\text{measured} = v\,\tau = \frac{2\pi r\,\tau}{N\,T}$$

The beam has a finite diameter $d$. It starts being blocked when the blade's leading edge
reaches the near side of the beam, and stops when the trailing edge leaves the far side, so

$$w_\text{blade} \approx w_\text{measured} - d$$

Note that $w$ depends only on the **ratio** τ/T, so the result shouldn't change with fan speed.
That gives us a check of the measurement (section 5).

### Which tutorial ideas are used where

| Step | Tutorial idea |
|---|---|
| Set fan speed, switch laser | On-demand AO (2a) |
| Live view for aligning the beam | Continuous AI loop / live plot (4a, 4b) |
| Measurement: laser switches on while AI records | Finite AO + AI synchronised with a start trigger (3) |
| Analysis | numpy on a finite AI recording (1b) |

### Wiring and safety

| Signal | DAQ pin | Notes |
|---|---|---|
| Fan speed | `AO0` | A DAQ AO pin can only supply a few mA, which is **not enough to run a fan**. Drive the fan through a MOSFET or transistor driver, or a motor driver board, so that 0–5 V on AO0 sets the speed. |
| Laser on/off | `AO1` | Use the laser module's TTL/enable input, or a transistor switch. 0 V = off, `LASER_ON_V` = on. |
| Photodiode | `AI0` (RSE) | Use a photodiode amplifier, or a photodiode with a load resistor, so you get a voltage. Ground goes to AI GND. |

⚠️ **Laser safety:** never look into the beam, and watch out for reflections off the fan blades.
Use a low-power module (class 2 or below) and keep the beam path below eye level.
⚠️ Keep fingers away from the spinning fan.

---
## Why a fan? The neuroscience connection

This setup is a small, safe model of two instruments that neuroscience labs use every day.

### A scanning (multiphoton) microscope

A two-photon microscope has **no camera**. It focuses a pulsed laser to a single point, moves
that point across the tissue with scan mirrors, and records the fluorescence from each
position with a single detector, a photomultiplier tube (PMT). The image only exists
because we know **where the beam was at each moment**: time is converted into position.

Our fan setup does the same thing the other way round. The beam stays still and the object
moves through it. We record the photodiode over time and use the known speed to turn time into distance:

$$\text{size} = \text{speed} \times \text{time}$$

That is exactly how a microscope's pixel size is set: µm per pixel = scan speed × time per pixel.
It's also why scan speed has to be known precisely. A **resonant scanner** (typically ~8 kHz)
moves sinusoidally, so it travels fastest in the middle of each line and slows at the edges,
and imaging software has to correct for that or the image is distorted at the edges. Measuring the fan's
rotation from the blade period T is like measuring a resonant scanner's actual frequency
from its sync output.

### An electrophysiology rig

Look at the photodiode trace and think "electrode": a steady baseline interrupted by brief,
stereotyped events. Finding the blade pulses is the same problem as **spike detection**:

* set a threshold relative to the baseline and the noise
* find the threshold crossings
* measure each event's **width** and the **interval** between events

The laser part maps onto **optogenetics**. There, a DAQ sends pulses to a laser or LED to
activate light-sensitive neurons, and a photodiode picking off some of the light is the standard
way to check that the light came on when it was commanded, and for as long.

### How the parts map

| This experiment | Two-photon microscope | Ephys / optogenetics rig |
|---|---|---|
| Laser switched by AO1 | Pockels cell controlling laser power, turned off ("blanked") between lines | Optogenetic laser/LED pulses |
| Fan blades moving through the beam | Scan mirrors moving the beam over the tissue | — |
| Photodiode on AI0 | PMT detecting fluorescence | Electrode + amplifier |
| Dark level (laser off) | PMT offset, subtracted from images | Baseline |
| Blade pulse | Beam passing over a labelled cell | A spike |
| Pulse width τ → blade width | Time per pixel → pixel size | Spike width |
| Blade period T | Line period of the scanner | Inter-spike interval → firing rate |
| AO start trigger starts AI | Mirror waveforms and PMT sampling locked together | Stimulus locked to recording |

The 🧠 notes through the notebook point out where each step shows up in a real experiment.

---
## 0. Setup

Leave `SIMULATE = True` to run the whole notebook without hardware. It generates a
realistic fake photodiode signal, so you can check the analysis first.
Set it to `False` once everything is wired up.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

import nidaqmx
from nidaqmx.constants import AcquisitionType, TerminalConfiguration

SIMULATE = True   # <-- set False to use the real DAQ

### Configuration

Everything you may need to change is in this one cell.

In [ ]:
# --- DAQ channels ---
DEV = "Dev1"
FAN_CHAN = f"{DEV}/ao0"
LASER_CHAN = f"{DEV}/ao1"
PD_CHAN = f"{DEV}/ai0"

# --- Output voltages ---
LASER_ON_V = 5.0       # voltage that switches the laser on
FAN_V = 4.0            # fan drive voltage for the main measurement

# --- Acquisition ---
RATE = 50_000          # samples/s. Check your device's maximum (e.g. USB-6001 is 20 kS/s)
DURATION = 1.0         # s of laser-on recording per measurement
DARK_TIME = 0.1        # s of laser-off recording at the start, to measure the dark level
SPIN_UP_TIME = 3.0     # s to wait for the fan to reach a steady speed

# --- The fan and the optics (measure these!) ---
N_BLADES = 7
BEAM_RADIUS_MM = 30.0  # distance from the fan's axis to the laser beam
BEAM_DIAMETER_MM = 1.0 # laser spot size at the fan

# --- Analysis ---
MIN_PULSE_S = 50e-6    # ignore "pulses" shorter than this (noise near the threshold)

### Fake data for `SIMULATE = True`

This models a fan whose speed is proportional to its drive voltage, with blades 15 mm wide
and a little noise. It has nothing to do with DAQmx. It's only there so the analysis can be
tested against a known answer.

In [ ]:
SIM_TRUE_BLADE_MM = 15.0
SIM_HZ_PER_VOLT = 8.0      # rotation frequency per volt of fan drive
SIM_DARK_V, SIM_LIGHT_V = 0.05, 2.0
rng = np.random.default_rng(0)


def fake_photodiode(laser_wave, fan_v):
    """Photodiode signal for a given laser AO waveform and fan voltage."""
    t = np.arange(len(laser_wave)) / RATE
    angle = 2 * np.pi * SIM_HZ_PER_VOLT * fan_v * t + rng.uniform(0, 2 * np.pi)
    blade_pitch = 2 * np.pi / N_BLADES
    blocked_angle = (SIM_TRUE_BLADE_MM + BEAM_DIAMETER_MM) / BEAM_RADIUS_MM
    blocked = (angle % blade_pitch) < blocked_angle
    laser_on = laser_wave > LASER_ON_V / 2
    light = np.where(laser_on & ~blocked, SIM_LIGHT_V, SIM_DARK_V)
    return light + rng.normal(0, 0.02, len(t))

---
## 1. Controlling the hardware

### Fan and laser: on-demand AO

Both outputs are in **one** AO task. Many devices can't run two AO tasks at the same time,
because the channels share one timing engine. A single task avoids that problem, and the
`write` takes one value per channel: `[fan, laser]`.

In [ ]:
def set_outputs(fan_v, laser_v):
    """Set fan and laser voltages immediately (on-demand AO). Values persist after return."""
    if SIMULATE:
        print(f"[sim] fan = {fan_v} V, laser = {laser_v} V")
        return
    with nidaqmx.Task() as task:
        task.ao_channels.add_ao_voltage_chan(FAN_CHAN, min_val=0, max_val=10)
        task.ao_channels.add_ao_voltage_chan(LASER_CHAN, min_val=0, max_val=10)
        task.write([fan_v, laser_v])


def all_off():
    set_outputs(0.0, 0.0)

Try it: switch the laser on with the fan stopped, and check the beam reaches the photodiode.

In [ ]:
set_outputs(fan_v=0.0, laser_v=LASER_ON_V)

### Live alignment view: continuous AI

While adjusting the laser and photodiode, it helps to watch the signal live. This is the
continuous loop from tutorial section 4b. Align for the **highest** signal with the fan stopped.
Then turn the fan by hand and check the signal drops close to zero when a blade is in the way.

In [ ]:
from IPython.display import clear_output, display


def live_view(seconds=10.0, chunk=5_000):
    """Plot the photodiode signal live for a while (continuous AI)."""
    if SIMULATE:
        print("[sim] live view needs hardware; skipping")
        return
    fig, ax = plt.subplots(figsize=(10, 3))
    with nidaqmx.Task() as task:
        task.ai_channels.add_ai_voltage_chan(
            PD_CHAN, terminal_config=TerminalConfiguration.RSE, min_val=-10, max_val=10
        )
        task.timing.cfg_samp_clk_timing(
            rate=RATE, sample_mode=AcquisitionType.CONTINUOUS, samps_per_chan=10 * chunk
        )
        task.start()
        t_end = time.time() + seconds
        while time.time() < t_end:
            data = np.array(task.read(number_of_samples_per_channel=chunk))
            ax.clear()
            ax.plot(np.arange(chunk) / RATE * 1000, data)
            ax.set_ylim(-0.5, 10)
            ax.set_xlabel("Time (ms)")
            ax.set_ylabel("Photodiode (V)")
            ax.set_title(f"Mean {data.mean():.3f} V")
            clear_output(wait=True)
            display(fig)
    plt.close(fig)


live_view(seconds=10)

---
## 2. The measurement: synchronised AO + AI

One measurement does the following:

1. Sets the fan to `fan_v` and waits for it to spin up (on-demand AO).
2. Runs a **finite AO task** that outputs a laser waveform: off for `DARK_TIME`, on for
   `DURATION`, then off again. The fan channel is held at `fan_v` throughout.
3. Runs a **finite AI task** on the photodiode, started by the AO start trigger
   (tutorial section 3). AI sample *i* is taken at the same moment as AO sample *i*.

Because the laser is off at the start of each recording, every recording includes its own
**dark level**. That makes the threshold robust to room light and amplifier offsets.

> **🧠 In the lab:** this is how a microscope acquires a frame. One finite AO task holds several
> channels (galvo X, galvo Y, Pockels cell) and the PMT recording is triggered from it.
> Putting the fan and laser in the same AO task, as here, is the same as putting the mirrors
> and laser power in one task: everything stays sample-locked. Recording a dark baseline at the
> start of each trial is also standard practice: a few hundred ms of PMT signal with the laser
> blanked, or a pre-stimulus baseline in ephys.

In [ ]:
def output_waveforms(fan_v):
    """AO waveforms (2 x n): fan held constant, laser off -> on -> off."""
    n_dark = int(DARK_TIME * RATE)
    n_on = int(DURATION * RATE)
    laser = np.concatenate([np.zeros(n_dark), np.full(n_on, LASER_ON_V), [0.0]])
    fan = np.full_like(laser, fan_v)  # last sample stays on the output -> fan keeps spinning
    return np.vstack([fan, laser])


def record(fan_v):
    """Run one synchronised laser + photodiode recording. Returns (t, photodiode, laser_wave)."""
    ao_data = output_waveforms(fan_v)
    n = ao_data.shape[1]
    t = np.arange(n) / RATE

    if SIMULATE:
        return t, fake_photodiode(ao_data[1], fan_v), ao_data[1]

    with nidaqmx.Task() as ao_task, nidaqmx.Task() as ai_task:
        ao_task.ao_channels.add_ao_voltage_chan(FAN_CHAN, min_val=0, max_val=10)
        ao_task.ao_channels.add_ao_voltage_chan(LASER_CHAN, min_val=0, max_val=10)
        ao_task.timing.cfg_samp_clk_timing(
            rate=RATE, sample_mode=AcquisitionType.FINITE, samps_per_chan=n
        )

        ai_task.ai_channels.add_ai_voltage_chan(
            PD_CHAN, terminal_config=TerminalConfiguration.RSE, min_val=-10, max_val=10
        )
        ai_task.timing.cfg_samp_clk_timing(
            rate=RATE, sample_mode=AcquisitionType.FINITE, samps_per_chan=n
        )
        ai_task.triggers.start_trigger.cfg_dig_edge_start_trig(f"/{DEV}/ao/StartTrigger")

        ao_task.write(ao_data, auto_start=False)
        ai_task.start()   # armed
        ao_task.start()   # triggers AI
        pd = np.array(ai_task.read(number_of_samples_per_channel=n, timeout=t[-1] + 5))
        ao_task.wait_until_done(timeout=5)

    return t, pd, ao_data[1]


def measure(fan_v):
    """Spin the fan up to fan_v, then record."""
    set_outputs(fan_v=fan_v, laser_v=0.0)
    if not SIMULATE:
        time.sleep(SPIN_UP_TIME)
    return record(fan_v)

In [ ]:
t, pd, laser_wave = measure(FAN_V)

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=False)
axes[0].plot(t, pd, lw=0.5)
axes[0].plot(t, laser_wave / LASER_ON_V * pd.max(), "r", alpha=0.5, label="laser on (scaled)")
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("Photodiode (V)")
axes[0].legend(loc="lower right")
axes[0].set_title("Whole recording")

zoom = (t > DARK_TIME + 0.1) & (t < DARK_TIME + 0.2)
axes[1].plot(t[zoom] * 1000, pd[zoom])
axes[1].set_xlabel("Time (ms)")
axes[1].set_ylabel("Photodiode (V)")
axes[1].set_title("Zoom: 100 ms with the laser on")
plt.tight_layout()
plt.show()

---
## 3. Finding the pulses

1. **Dark level:** the mean signal while the laser was off.
2. **Light level:** the unblocked signal with the laser on. We use the 95th percentile so a few noisy samples don't skew it.
3. **Threshold:** halfway between the two. Below the threshold means *blocked*.
4. **Edges:** where *blocked* changes from False to True (blade arrives) or True to False (blade leaves).
5. Pair each arrival with the next departure and discard incomplete pulses at either end.

Measuring at the half-way level also means the beam diameter enters symmetrically: each
edge is detected when the blade covers half of the beam.

> **🧠 In the lab:** this is threshold-crossing **spike detection**. In ephys the events are
> small compared with the noise, so the threshold is set from the noise rather than halfway to
> a known level. It is typically 4–5 × σ, with the noise σ estimated robustly as
> `median(|x|) / 0.6745` so the spikes themselves don't inflate it (Quiroga et al., 2004).
> `MIN_PULSE_S` plays the role of discarding events too short to be real, and the
> *hysteresis* idea at the end of the notebook is a common fix for noisy threshold crossings.

In [ ]:
def levels(pd, laser_wave):
    """Return (dark, light, laser_on_mask) from one recording."""
    laser_on = laser_wave > LASER_ON_V / 2
    dark = pd[~laser_on].mean()
    light = np.percentile(pd[laser_on], 95)
    return dark, light, laser_on


def find_pulses(signal, threshold, rate, min_width_s=MIN_PULSE_S):
    """Find intervals where signal < threshold.

    Returns (starts, widths) in seconds. Only complete pulses are returned.
    """
    blocked = (signal < threshold).astype(int)
    edges = np.diff(blocked)
    starts = np.flatnonzero(edges == 1) + 1
    ends = np.flatnonzero(edges == -1) + 1

    if starts.size == 0 or ends.size == 0:
        return np.array([]), np.array([])

    ends = ends[ends > starts[0]]          # drop a pulse that was already underway
    n = min(starts.size, ends.size)        # drop a pulse still underway at the end
    starts, ends = starts[:n], ends[:n]

    widths = (ends - starts) / rate
    keep = widths >= min_width_s           # crude de-glitching
    return starts[keep] / rate, widths[keep]

In [ ]:
dark, light, laser_on = levels(pd, laser_wave)
threshold = (dark + light) / 2
print(f"dark = {dark:.3f} V, light = {light:.3f} V, threshold = {threshold:.3f} V")

if light - dark < 0.1:
    print("⚠️  Very little contrast between light and dark: check the alignment with live_view()")

# Analyse only the laser-on part of the recording
starts, widths = find_pulses(pd[laser_on], threshold, RATE)
print(f"Found {len(widths)} pulses")

### Did the laser come on when we told it to?

In optogenetics you want to know the delay between *commanding* the light and the light
*arriving*, and whether the pulse lasted as long as intended. We recorded both the command
(our own AO waveform) and the light (the photodiode) on the same clock, so the delay is just
the number of samples between them. If a blade happens to be blocking the beam at that
moment, the delay looks longer, so run it a few times.

In [ ]:
commanded_on = np.flatnonzero(laser_on)[0]
measured_on = commanded_on + np.flatnonzero(pd[commanded_on:] > threshold)[0]
print(f"Laser light arrived {(measured_on - commanded_on) / RATE * 1e3:.3f} ms after the command")

Check the detection by eye: the shaded regions should line up with the dips.

In [ ]:
seg_t = np.arange(laser_on.sum()) / RATE
seg = pd[laser_on]
show = seg_t < 0.1

plt.figure(figsize=(10, 3))
plt.plot(seg_t[show] * 1000, seg[show], lw=0.8)
plt.axhline(threshold, color="k", ls=":", label="threshold")
for s, w in zip(starts, widths):
    if s < 0.1:
        plt.axvspan(s * 1000, (s + w) * 1000, color="orange", alpha=0.3)
plt.xlabel("Time since laser on (ms)")
plt.ylabel("Photodiode (V)")
plt.legend(loc="lower right")
plt.title("Detected blade pulses (first 100 ms)")
plt.show()

---
## 4. Pulse width → blade width

* **Blade period** T: the time between successive pulse starts.
* **Rotation frequency** f = 1 / (N·T).
* **Blade speed at the beam** v = 2πrf.
* **Blade width** = v·τ − beam diameter.

We compute a width for every pulse. Taking the median is robust to the odd bad detection.

In [ ]:
def blade_widths(starts, widths, n_blades=N_BLADES, radius_mm=BEAM_RADIUS_MM,
                 beam_mm=BEAM_DIAMETER_MM):
    """Per-pulse blade width (mm) and the rotation frequency (Hz)."""
    if len(starts) < 2:
        raise ValueError("Need at least 2 pulses. Is the fan spinning and the beam aligned?")
    blade_period = np.median(np.diff(starts))
    rot_freq = 1 / (n_blades * blade_period)
    speed_mm_s = 2 * np.pi * radius_mm * rot_freq
    return speed_mm_s * widths - beam_mm, rot_freq


w_mm, rot_freq = blade_widths(starts, widths)

print(f"Rotation:      {rot_freq:.2f} Hz  ({rot_freq * 60:.0f} rpm)")
print(f"Pulse width:   {np.median(widths) * 1e3:.3f} ms (median)")
print(f"Blade width:   {np.median(w_mm):.2f} mm  "
      f"(median, spread ±{np.std(w_mm):.2f} mm over {len(w_mm)} pulses)")
if SIMULATE:
    print(f"[sim] true blade width was {SIM_TRUE_BLADE_MM} mm")

> **🧠 In the lab:** `speed × time = distance` is the whole calculation behind a scanning
> microscope's pixel size. For example, a resonant scanner at 8 kHz that sweeps 500 µm in roughly
> half of each 125 µs period, sampled into 512 pixels, gives about 1 µm per pixel and about 0.1 µs per
> pixel. In ephys, the same per-event numbers become **spike width** and **firing rate**
> (1 / inter-spike interval). Spike width is often used to tell narrow-spiking, putative
> fast-spiking interneurons apart from broad-spiking, putative pyramidal cells.

### Each blade separately

Pulse *k* comes from blade *k mod N*, so grouping by that index shows whether the blades
differ from one another. Which group is "blade 1" is arbitrary, because we don't know which blade passed first.

In [ ]:
blade_index = np.arange(len(w_mm)) % N_BLADES
per_blade = [w_mm[blade_index == b] for b in range(N_BLADES)]

plt.figure(figsize=(6, 3))
plt.boxplot(per_blade)
plt.xlabel("Blade (arbitrary numbering)")
plt.ylabel("Width (mm)")
plt.title("Width of each blade")
plt.show()

If one blade's box is clearly offset from the others, check that `N_BLADES` is correct.
If it's wrong, the per-blade groups mix different blades together and the rotation frequency is off by a factor.

> **🧠 In the lab:** sorting events by "which blade made them" is a very simple version of
> **spike sorting**, i.e. working out which neuron each spike on an electrode came from. Here it's
> easy because the blades arrive in a fixed order. With real neurons you group spikes by waveform
> shape instead, and getting the number of groups wrong causes the same kind of mixing.

---
## 5. Check: does the width depend on fan speed?

It shouldn't. A faster fan makes τ and T shorter in the same proportion. Repeat the
measurement at several fan voltages: the rpm should change, the width should not.
If the width *does* drift with speed, likely causes are:

* a photodiode/amplifier that is too slow, which rounds off the edges at high speed
* a sample rate that is too low for the shortest pulses
* fan speed drifting during the recording (wait longer to spin up)

> **🧠 In the lab:** this is a **calibration** check, and microscopes need the same thing. Imaging
> software lets you change zoom and scan speed, and the pixel size has to stay correct at every setting.
> It's checked by imaging something of known size, such as a calibration grid or fluorescent
> beads, at several settings, the way we check the blade width at several speeds. A detector
> or amplifier that is too slow (low bandwidth) blurs fast scans in the same way it would round
> off the blade edges here.

In [ ]:
FAN_VOLTAGES = [2.0, 3.0, 4.0, 5.0]
results = []

for v in FAN_VOLTAGES:
    t, pd, laser_wave = measure(v)
    dark, light, laser_on = levels(pd, laser_wave)
    starts, widths = find_pulses(pd[laser_on], (dark + light) / 2, RATE)
    w_mm, f = blade_widths(starts, widths)
    results.append((v, f * 60, np.median(w_mm), np.std(w_mm)))
    print(f"{v:4.1f} V: {f * 60:6.0f} rpm, width {np.median(w_mm):6.2f} ± {np.std(w_mm):.2f} mm")

all_off()

In [ ]:
volts, rpm, width, spread = np.array(results).T

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
ax1.plot(volts, rpm, "o-")
ax1.set_xlabel("Fan drive (V)")
ax1.set_ylabel("Speed (rpm)")
ax1.set_title("Fan speed")

ax2.errorbar(rpm, width, yerr=spread, fmt="o", capsize=4)
ax2.axhline(np.mean(width), color="k", ls=":")
ax2.set_xlabel("Speed (rpm)")
ax2.set_ylabel("Blade width (mm)")
ax2.set_title("Blade width vs speed (should be flat)")
plt.tight_layout()
plt.show()

print(f"Blade width over all speeds: {np.mean(width):.2f} ± {np.std(width):.2f} mm")

---
## 6. Switch everything off

In [ ]:
all_off()

## Things to try

* **Measure with a ruler** and compare. Fan blades are angled (pitched), so what the beam measures
  is the blade's width along the direction of rotation at radius r, not along its surface.
* **Move the beam to a different radius.** Many blades get wider towards the tip.
* **Better edge timing:** interpolate between the two samples either side of each threshold
  crossing, instead of using whole samples. This matters when τ is only a few samples long.
* **Hysteresis:** use two thresholds (e.g. 40 % and 60 %) instead of one, so noise near the
  threshold can't create extra edges. This is more robust than `MIN_PULSE_S`.
* **Measure the beam diameter** by moving a razor blade across the beam on a translation stage
  while watching `live_view()`. This "knife-edge" method is also used to measure laser spot sizes
  on real microscopes.
* **Make an image:** record for one full rotation and reshape the signal into rows, one per
  rotation, over several rotations. You've built a (very slow) scanning imager. Compare it with
  how a microscope reshapes the PMT signal into a frame.
* **Optogenetic-style pulse train:** change `output_waveforms` so the laser outputs 10 × 5 ms
  pulses at 20 Hz instead of one long step, and use `find_pulses` on the photodiode (with the
  fan stopped) to check each pulse's actual duration.